In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Tue Mar 26 13:41:03 2024

@author: methvenr
"""

In [1]:
# Import libraries
import pandas as pd
pd.options.mode.copy_on_write = True
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Import data
df = pd.read_csv('C:/Users/methvenr/Documents/ratings_consolidated.csv', sep = '|')

In [3]:
# Load user data
data = df[['learner_employee_id','training_id','rating']].copy()
data = data.dropna()

In [4]:
# Create a pivot table of users against videos
user_ratings = data.pivot_table(index='learner_employee_id', columns='training_id', values='rating')

In [5]:
# Fill NaN values with 0 
user_ratings = user_ratings.fillna(0)

In [6]:
# Calculate similarity between users
user_sim = cosine_similarity(user_ratings)

In [7]:
# Create a looukp of learner ID that has a matching index with pivot table
emps = pd.DataFrame(data.learner_employee_id.unique())
emps = emps.rename(columns = {0: 'learner_employee_id'})
emps = emps.sort_values(by = ['learner_employee_id'])
emps = emps.reset_index(drop=True)
emps = emps.reset_index()

In [8]:
# Create empty list for output
recs = []

In [9]:
for index, row in emps.iterrows():
    
    target_user_id = emps.index.get_loc(emps[emps['learner_employee_id'] == (row['learner_employee_id'])].index[0])
    
    similar_users = user_sim[target_user_id].argsort()[::-1][1:501]

    # Get unrated videos for the target user - similar_users is based on index, need to find the learner_id that corresponds with the index number
    user_ratings_filtered = user_ratings.iloc[target_user_id].to_frame().reset_index()
    unrated_videos = user_ratings_filtered[user_ratings_filtered[(row['learner_employee_id'])]==0]['training_id']

    # Data wrangle for the following functions
    similar_users_df = pd.DataFrame(similar_users)
    similar_users_df = similar_users_df.rename(columns = {0: 'index'})

    similar_users_final = pd.merge(left = similar_users_df, right = emps, how = 'inner')

    for training_id in unrated_videos:
        similar_user_ratings = user_ratings.loc[similar_users_final['learner_employee_id']][training_id]
        similar_user_ratings = pd.DataFrame(similar_user_ratings)
        similar_user_ratings = similar_user_ratings.loc[(similar_user_ratings!=0).any(axis=1)]
        mean_rating = pd.DataFrame(similar_user_ratings.mean())
        recs.append(((row['learner_employee_id']),training_id, mean_rating.loc[:,0].item()))

In [10]:
recs_df = pd.DataFrame(recs).dropna().reset_index(drop=True)
recs_df = recs_df.rename(columns={0: "emplid", 1: "training_id", 2: "rating"}).sort_values(['emplid','rating'], ascending=[True, False])
recs_final = recs_df.groupby('emplid').head(10)
recs_final['rank'] = recs_final.groupby('emplid')['rating'].rank(method='min', ascending=False)
recs_final = recs_final.reset_index(drop=True)

In [11]:
recs_final.to_csv('C:/Users/methvenr/Documents/content_recs_conda.csv', index=False)